# 09.5 — LLM-as-a-Judge Evaluation Flow (TREC-Style Ground Truth)
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:** `data/processed/books.csv`  
**Output:** `data/eval/qrels.json` (TREC-style query-to-document relevance map)  

Notebook này minh họa quy trình tạo bộ dữ liệu đánh giá chất lượng cao (Ground Truth) sử dụng phương pháp **LLM-as-a-Judge** tích hợp từ mã nguồn hệ thống mới (`src/chains/` và `src/services/`):
1. **LLM**: Sử dụng OpenAI (`gpt-4o-mini`) để sinh câu hỏi tự động và đánh giá độ liên quan.
2. **Cơ chế**: Không chỉ map 1-1 đơn giản, mà quét qua một nhóm ứng viên (candidate pool) và nhờ LLM chấm điểm độ liên quan ở 3 mức độ (0: không liên quan, 1: liên quan một phần, 2: rất liên quan) để tạo file `qrels.json` chuẩn công nghiệp.

## 1. Khởi tạo cấu hình và Load dữ liệu sách

In [ ]:
import pandas as pd
import json
from pathlib import Path

DATA_PATH = Path('data/processed/books.csv')
QRELS_PATH = Path('data/eval/qrels.json')

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} books from dataset.")

## 2. Giải thích Luồng sinh Query bằng LLM
Mã nguồn trong `src/chains/query_generation_chain.py` định nghĩa prompt yêu cầu LLM đóng vai trò người dùng tìm kiếm sách. Từ cốt truyện sách (`description`), LLM sinh ra 3 mức độ truy vấn từ chi tiết tới mơ hồ:
- **Query 1 (Broad/Short)**: Truy vấn ngắn, mang tính chủ đề (ví dụ: `father-daughter relationship`).
- **Query 2 (Medium)**: Mô tả cốt truyện ở mức trung bình.
- **Query 3 (Detailed)**: Truy vấn dài, chi tiết mô tả cốt truyện nhưng không chứa tên sách/tác giả.

In [ ]:
# Minh họa cấu trúc prompt sinh query từ src/chains/query_generation_chain.py
system_prompt = """
You are an expert evaluator for a book search engine. 
Given a book's metadata and description, your task is to generate 3 realistic search queries 
that a user might type into a search bar if they were looking for this book but didn't remember the title or author.
"""
print("LLM Query Generation Prompt configured! ✓")

## 3. Giải thích Luồng đánh giá Relevance (LLM-as-a-Judge)
Trong phương pháp đánh giá hệ thống tìm kiếm hiện đại (như TREC hay MS MARCO):
1. Với mỗi query sinh ra, hệ thống chạy mô hình Dense Retriever để lấy ra Top 20 cuốn sách có độ tương đồng vector cao nhất.
2. LLM (`gpt-4o-mini`) sẽ đọc cặp `(Query, Candidate Book)` và chấm điểm theo Rubric sau:
   * **0 - NOT RELEVANT**: Cuốn sách hoàn toàn không liên quan đến ý định tìm kiếm của câu truy vấn.
   * **1 - SOMEWHAT RELEVANT**: Cuốn sách liên quan một phần (chung thể loại, chung chủ đề lớn nhưng không khớp chi tiết).
   * **2 - HIGHLY RELEVANT**: Cuốn sách trùng khớp hoàn hảo với nội dung người dùng đang tìm.
3. Sách được chấm 1 hoặc 2 điểm sẽ được đưa vào danh sách `relevant_isbns` trong file nhãn đúng `qrels.json`.

In [ ]:
# Minh họa Rubric chấm điểm trong src/chains/relevance_judge_chain.py
relevance_rubric = """
Scoring rubric:
  0 - NOT RELEVANT: The book does not match the query's intent, themes, or subject.
  1 - SOMEWHAT RELEVANT: The book partially matches the query (e.g., shares a theme/genre but not specific topic).
  2 - HIGHLY RELEVANT: The book directly matches the query's intent.
"""
print("LLM Relevance Judge Rubric loaded! ✓")

## 4. Xem Dữ liệu Đánh giá đã sinh ra (`qrels.json`)

In [ ]:
# Đọc và hiển thị 3 phần tử đầu tiên của qrels.json đã được sinh từ trước
with open(QRELS_PATH, 'r', encoding='utf-8') as f:
    qrels_data = json.load(f)

print(f"Tổng số câu hỏi đánh giá trong qrels.json: {len(qrels_data)}")
print("\nVí dụ 3 bản ghi đầu tiên:")
print(json.dumps(qrels_data[:3], indent=2, ensure_ascii=False))

## 5. Kết luận
Nhờ cách tiếp cận mới này, tập đánh giá không còn bị giới hạn bởi việc bắt buộc tìm đúng cuốn sách ban đầu (1-to-1 match), mà chấp nhận mọi cuốn sách có nội dung tương tự được đề xuất bởi hệ thống, giúp kết quả benchmark khách quan và sát với thực tế sử dụng hơn.